### Feature Extraction - Multiclass (Using InceptionV2 with Callbacks)

**AIM**: Build and train an image classifier to detect images from different animal species using a pre-trained InceptionV2 model with models from Keras Hub

### Objectives
- Install TensorFlow Hub
- Data visualisation
- Data preprocessing and image augmentation
- Use a pre-trained CNN Model for transfer learning (dynamic feature extraction)
- Base model runs every epoch and supports data augmentation and fine-tuning
- Compile and train the model
- Add early stopping callback (optional)
- Save and load model
- Model evaluation
- Make prediction on new data

### Prerequisite
- Google colab or Jupyter notebook
- animal-image-classification-dataset
- TensorFlow2

### Check if TensorFlow, NumPy, Pandas and Matplotlib are installed

In [1]:
!pip show tensorflow
!pip install tensorflow-hub
!pip install tf-keras

Name: tensorflow
Version: 2.19.0
Summary: TensorFlow is an open source machine learning framework for everyone.
Home-page: https://www.tensorflow.org/
Author: Google Inc.
Author-email: packages@tensorflow.org
License: Apache 2.0
Location: /home/agbor/anaconda3/envs/tf/lib/python3.11/site-packages
Requires: absl-py, astunparse, flatbuffers, gast, google-pasta, grpcio, h5py, keras, libclang, ml-dtypes, numpy, opt-einsum, packaging, protobuf, requests, setuptools, six, tensorboard, tensorflow-io-gcs-filesystem, termcolor, typing-extensions, wrapt
Required-by: spektral, tensorflow-gnn, tensorflow-text, tf_keras


In [2]:
# !pip show numpy

In [3]:
# !pip show pandas

In [4]:
!pip show matplotlib

Name: matplotlib
Version: 3.7.5
Summary: Python plotting package
Home-page: https://matplotlib.org
Author: John D. Hunter, Michael Droettboom
Author-email: matplotlib-users@python.org
License: PSF
Location: /home/agbor/anaconda3/envs/tf/lib/python3.11/site-packages
Requires: contourpy, cycler, fonttools, kiwisolver, numpy, packaging, pillow, pyparsing, python-dateutil
Required-by: captum, catboost, missingno, pandas-profiling, phik, pycaret, pyod, pytorch-tabular, SALib, scikit-plot, seaborn, wordcloud, ydata-profiling, yellowbrick


In [5]:
pip shw tensorflow_hub

ERROR: unknown command "shw" - maybe you meant "show"
Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import tensorflow as tf
import tensorflow_hub as hub

2026-02-19 19:56:07.575724: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771527367.590996   36745 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771527367.595329   36745 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771527367.606792   36745 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771527367.606810   36745 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771527367.606812   36745 computation_placer.cc:177] computation placer alr

In [7]:
import sys
import random
import pathlib
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [8]:
# Set seed for reproducibility

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

In [9]:
# Check for GPU
!nvidia-smi

Thu Feb 19 19:56:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Quadro RTX 4000                Off |   00000000:01:00.0  On |                  N/A |
| N/A   47C    P8              9W /  110W |      75MiB /   8192MiB |     29%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [10]:
# Check if TensorFlow can detect any GPU
gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print(f"GPUs available: {len(gpus)}")

    for gpu in gpus:
        print(f"- {gpu}")
else:
    print("No GPUs found.")

GPUs available: 1
- PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


In [11]:
gpus = tf.config.list_physical_devices("GPU")
gpus

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [12]:
# Get the total number of GPUs and names
if gpus:
    print(f"Total number of GPUs: {len(gpus)}")
    
    for gpu in gpus:
        print(f"GPU Name: {gpu}")
else:
    print("No GPU available")

Total number of GPUs: 1
GPU Name: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


In [13]:
# Check TensorFlow version
print(f"TensorFlow Version: {tf.__version__}")

TensorFlow Version: 2.19.0


In [14]:
# CReate a director called "models" to store model files during training

if not os.path.isdir("models"):
    os.mkdir("models")

In [15]:
## Set the base path
base_dir = "../../datasets/dog_vs_cats"
base_dir = pathlib.Path(base_dir)
base_dir

PosixPath('../../datasets/dog_vs_cats')

In [16]:
# Train directory
train_dir = base_dir / "train"
train_dir

PosixPath('../../datasets/dog_vs_cats/train')

In [17]:
# Validation directory
test_dir = base_dir / "test"
test_dir

PosixPath('../../datasets/dog_vs_cats/test')

In [18]:
# Validation directory
validation_dir = base_dir / "validation"
validation_dir

PosixPath('../../datasets/dog_vs_cats/validation')

In [19]:
IMAGE_HEIGHT = 128
IMAGE_WIDTH = 128
BATCH_SIZE = 32
EPOCHS = 10

In [20]:
# Load the training dataset
# Using `image_dataset_from_directory (recommended and stable)
# Instead of using the 'ImageDataGenerator()' class

train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(IMAGE_HEIGHT, IMAGE_WIDTH),
    batch_size=BATCH_SIZE,
    seed=SEED,
)

Found 20000 files belonging to 2 classes.


I0000 00:00:1771527371.013633   36745 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6555 MB memory:  -> device: 0, name: Quadro RTX 4000, pci bus id: 0000:01:00.0, compute capability: 7.5


In [21]:
# Load the validation dataset

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    validation_dir,
    image_size=(IMAGE_HEIGHT, IMAGE_WIDTH),
    batch_size=BATCH_SIZE,
    seed=SEED,
)

Found 5000 files belonging to 2 classes.


In [22]:
# Load the validation dataset

test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(IMAGE_HEIGHT, IMAGE_WIDTH),
    batch_size=BATCH_SIZE,
    seed=SEED,
)

Found 12461 files belonging to 2 classes.


In [23]:
# Get the class names
class_names = train_dataset.class_names
number_classes = len(class_names)

print(f"Number of classes: {number_classes}")
print(f"Class names: {list(class_names)}")

Number of classes: 2
Class names: ['cats', 'dogs']


### Sanity Check: Visualise dataset samples 
- Confirms correct labels
- Detects corrupt images
- Verifies colour channels, orientation, resolution

In [24]:
# Verify (test)

train_dataset.take(1)

<_TakeDataset element_spec=(TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

In [25]:
# Capture exactly ONE batch and store it in memory
# This "freezes" the random augmentations for this specific set
for images, labels in train_dataset.take(1):
    fixed_images = images.numpy()
    fixed_labels = labels.numpy()

2026-02-19 19:56:12.402054: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [ ]:
# Visualisation
# No matter how many times you run this cell, the images won't change.
plt.figure(figsize=(12, 12))
for i in range(16):
    ax = plt.subplot(4, 4, i + 1)
    plt.imshow(fixed_images[i].astype("uint8"))
    plt.title(class_names[fixed_labels[i]])
    plt.axis("off")

### Performance Optimisation

Using `tf.data.AUTOTUNE` and the specific method below creates an asynchronous pipeline—basically, it allows the CPU to prepare the next batch while the GPU is still processing the current one.

In [ ]:
# AUTOTUNE allows TensorFlow to dynamically adjust the resource allocation 
# (like CPU threads) based on your hardware's current workload at runtime.
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = (
    train_dataset
    # Keep the data in memory (RAM) after the first epoch.
    # This prevents the CPU from having to re-read and re-decode JPEGs 
    # from the slow hard drive during every subsequent training round.
    .cache()

    # Maintain a buffer of 1,000 samples and randomly pulls from it.
    # This ensures the model doesn't learn the order of the files, 
    # but rather the features of the images.
    .shuffle(1000)

    # Overlaps the data preprocessing and model execution.
    # While the GPU is training on the current batch, the CPU is already 
    # preparing the next batch in the background. This eliminates "GPU starvation."
    .prefetch(buffer_size=AUTOTUNE)
)

In [ ]:
validation_dataset = (
    validation_dataset
    .cache()
    .prefetch(buffer_size=AUTOTUNE)
)

In [ ]:
test_dataset = (
    test_dataset
    .cache()
    .prefetch(buffer_size=AUTOTUNE)
)

### Configure Callbacks

In [ ]:
import datetime

model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath="models/InceptionV2.keras",
    monitor="val_accuracy",
    save_best_only=True,
    # verbose=1
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True,
    verbose=1
)

reduce_learning_rate = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.3,
    patience=5,
    verbose=1
)

model_path = pathlib.Path("models")
log_dir = model_path / "InceptionV2" / datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard = tf.keras.callbacks.TensorBoard(
    log_dir=log_dir,
)

model_name = "InceptionV2"
file_name = model_path / f"{model_name}training_metrics.csv"
csv_logger = tf.keras.callbacks.CSVLogger(
    filename=file_name,
    separator=",",
    append=False,
)

callbacks = [model_checkpoint, tensorboard, csv_logger, early_stopping, reduce_learning_rate]

### Data Augmentation (Modern Keras Layers)

In [ ]:
# Data Augmentation

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
])

### Model Input Shape

In [ ]:
INPUT_SHAPE = (IMAGE_HEIGHT, IMAGE_WIDTH) + (3,)
INPUT_SHAPE

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
from tensorflow.keras import layers, models

In [ ]:
### Function to build model

def buildModel(model_url, num_classes=1000, input_shape=(224, 224, 3)):

    """
    Takes a TensorFlow Hub (or Keras Hub) URL and creates a Keras Sequential class with it.

    Args:
        model_url (str): A TensorFlow Hub feature extration URL
        num_lasses (int): Number of output neurons in the output layer 
            (1 == binary, num_classes == multiclass classification),
            This numbe rshould be equal to number of target classes.      

    Returns:
        An unompiled Keras Sequential model with model_url as feature extractor
        layer and Dense output layer with num_classes output neurons.
    """

    # Download the pretrained model and save it as  a Keras layer
    feature_extractor_layer = hub.KerasLayer(model_url,
                                             trainable=False,
                                             name="feature_extration_layer",
                                             input_shape=input_shape) # Freeze the already learnt patterns
    
    # Create the classification head
    model = tf.keras.Sequential([
        feature_extractor_layer,
        layers.Dense(num_classes, 
                     activation=["sigmoid" if num_classes==1 else "softmax"][0],
                     name="output_layer")
    ])
        
    return model


### Creating the InceptionV2 Model

In [ ]:
inceptionv2_url = "https://www.kaggle.com/models/google/inception-v2/TensorFlow2/feature-vector/2"

name = "Inception_model"
model = buildModel(inceptionv2_url, num_classes=1, input_shape=INPUT_SHAPE)
model.summary()

In [ ]:
num_classes = 1
activation="sigmoid" if num_classes==1 else "softmax"
activation

### Build and Compile the Model

In [ ]:
# Compile the model

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-7),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

### Train the model to learn patterns from the images

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

### Plot Training Curves (Learning Dynamics)

In [ ]:
file_name = model_path / f"{model_name}_training_metrics.csv"
file_name

In [ ]:
log_dir

In [ ]:
def plot_learning_curves(history):
    acc = history.history["accuracy"]
    val_acc = history.history["val_accuracy"]
    loss = history.history["loss"]
    val_loss = history.history["val_loss"]

    epochs_range = range(len(acc))


    plt.figure(figsize=(18, 7))

    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label="Training Accuracy")
    plt.plot(epochs_range, val_acc, label="Validation Accuracy")
    plt.legend()
    plt.title("Accuracy")

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label="Training Loss")
    plt.plot(epochs_range, val_loss, label="Validation Loss")
    plt.legend()
    plt.title("Loss")

    plt.show()


In [ ]:
plot_learning_curves(history)

### Generate Predictions (TensorFlow to NumPy)

In [ ]:
loss, accuracy = model.evaluate(validation_dataset)

print(f"Model Loss: {loss:.2f}")
print(f"Model Accuracy: {accuracy:.2f}")

In [ ]:
y_true = []
y_pred = []
y_prob = []

for images, labels in test_dataset:
    preds = model.predict(images, verbose=1) # Try verbose=0
    y_prob.append(preds)
    y_pred.append(preds >= 0.5)
    y_true.append(labels.numpy())

In [ ]:
len(y_true), len(y_pred), len(y_prob)

In [ ]:
y_true = np.concatenate(y_true)
y_pred = np.concatenate(y_pred)
y_prob = np.concatenate(y_prob)

### Core Classification Metrics (from scikit-learn)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)



print(f"Accuracy: {accuracy_score(y_true, y_pred)}")
print(f"Precision (macro): {precision_score(y_true, y_pred)}")
print(f"Recall (macro): {recall_score(y_true, y_pred):.2f}")
print(f"F1-score (macro): {f1_score(y_true, y_pred):.2f}")

In [ ]:
# Full Classification Report

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    digits=4
))


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(6, 5))

# Create custom annotations showing both count and percentage
annot_text = np.empty_like(cm).astype(str)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        annot_text[i, j] = f'{cm[i, j]}\n({cm_norm[i, j]:.1%})'

sns.heatmap(cm, 
            cmap="Blues", 
            cbar=False, 
            annot=annot_text, 
            fmt='',  # fmt='' for string annotations
            xticklabels=class_names,
            yticklabels=class_names,
            
            # kwargs for annotation text
            annot_kws={"fontsize": 12, "fontweight": "bold"},)  

plt.title("Confusion Matrix (Counts and Percentages)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(rotation=45, ha="right")

# Adjust layout to prevent text cutoff
plt.tight_layout()  
plt.show()

In [ ]:
# Saving the Model

model.save("dogs_cat_InceptionV2.keras")

**This resulted in a poor model. We will use transfer learning to see how to improve the model using any of the model which is already doing well in general image processing tasks.**